# Synthetic Dataset of Driver Telematics 

## Import packages

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from typing import Callable
import dice_ml
import xgboost as xgb
from numpy.random import seed
seed(42)

# Fix seed
import random
random.seed(42)

In [2]:
import os
os.getcwd()

'/Users/agathefernandesmachado/Documents/PhD/counterfactuals_insurance/scripts_python'

## Import dataset

In [3]:
df_train = pd.read_csv("../out/flood_df_train.csv")
df_calib = pd.read_csv("../out/flood_df_calib.csv")
df_test = pd.read_csv("../out/flood_df_test.csv")

In [4]:
df_train.head()

,basement,elevated_building,contents_location,nb_floors,non_profit,occupancy,primary_residence,damage_cause,rental,building_age,flood_zone,Y
0,0,False,Lowest_floor_and_above,1,False,Building_non_residential,False,4,False,51,AE_zone,0.003150
1,0,False,Lowest_floor_only,1,False,Single_family_residential,False,1,False,53,AE_zone,0.364012
2,0,False,Lowest_floor_only,2,False,Single_family_residential,True,4,False,61,AE_zone,0.339609
3,0,False,Lowest_floor_only,1,False,Building_non_residential,False,1,False,53,AE_zone,0.022751
4,0,False,Lowest_floor_and_above,2,False,Building_non_residential,False,4,False,97,AE_zone,0.126447


In [5]:
df_train.shape

(34698, 12)

## Severity model: XGBoost

In [6]:
xgb_model = xgb.XGBRegressor()
xgb_model.load_model("../out/flood_xgb_model.json")

In [7]:
# Load exactly the matrix used by R
X_train = pd.read_csv("../out/flood_xgb_train_matrix.csv")
X_test = pd.read_csv("../out/flood_xgb_test_matrix.csv")
X_test.head()

,basement0,basement1,basement2,basement4,elevated_buildingTRUE,contents_locationLowest_floor_and_above,contents_locationUpper_floors,contents_locationMobile_home,nb_floors2,nb_floors3,...,damage_cause3,damage_cause4,damage_cause7,damage_cause9,damage_causeA,rentalTRUE,building_age,flood_zoneLow_moderate_zone,flood_zoneOther_A_zone,flood_zoneV_zone
0,1,0,0,0,1,1,0,0,0,1,...,0,1,0,0,0,0,6,0,0,0
1,1,0,0,0,1,0,0,1,0,0,...,0,1,0,0,0,0,62,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,59,0,0,1
3,1,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,77,0,0,0
4,1,0,0,0,1,1,0,0,0,1,...,0,1,0,0,0,0,4,0,0,0


In [8]:
# Predictions in Python
pred_test = xgb_model.predict(X_test)
pred_test[:5]

array([0.01320685, 0.39534602, 0.3589253 , 0.35351107, 0.01371206],
      dtype=float32)

In [9]:
# 1st Query instance
idx_query = 19
query_instance = X_test.iloc[[idx_query]]
df_test.iloc[[idx_query]]

,basement,elevated_building,contents_location,nb_floors,non_profit,occupancy,primary_residence,damage_cause,rental,building_age,flood_zone,Y
19,0,False,Lowest_floor_only,1,False,Single_family_residential,True,1,False,64,AE_zone,0.528804


In [10]:
df_test.columns

Index(['basement', 'elevated_building', 'contents_location', 'nb_floors',
       'non_profit', 'occupancy', 'primary_residence', 'damage_cause',
       'rental', 'building_age', 'flood_zone', 'Y'],
      dtype='str')

In [11]:
pred_test[idx_query] # Same prediction as in R

np.float32(0.45821247)

In [12]:
#def preprocess(df):
#    df = df.copy()

    # Set categorical variables
#    categorical_cols = [
#        "basement",
#        "elevated_building",
#        "contents_location",
#        "nb_floors",
#        "non_profit",
#        "occupancy",
#        "primary_residence",
#        "damage_cause",
#        "rental",
#        "flood_zone",
#    ]

#    for col in categorical_cols:
#        df[col] = df[col].astype("category")

#    return df

In [13]:
#def make_xgb_df(df):
#    df = preprocess(df)

    # Equivalent to model.matrix(~ 0 + ., data = df)
#    xgb_df = pd.get_dummies(
#        df,
#        columns=[
#            "basement",
#            "elevated_building",
#            "contents_location",
#            "nb_floors",
#            "non_profit",
#            "occupancy",
#            "primary_residence",
#            "damage_cause",
#            "rental",
#            "flood_zone",
#        ],
#        dtype=int
#    )

    # Equivalent to select(-Y)
#    xgb_df = xgb_df.drop(columns=[
#        "elevated_building_False", "contents_location_Lowest_floor_only", "nb_floors_1", "non_profit_False",
#        "occupancy_Building_non_residential", "primary_residence_False", "damage_cause_0", "rental_False",
#        "flood_zone_AE_zone"
#        ])
#    xgb_df[["damage_cause_7", "damage_cause_9", "damage_cause_A"]] = 0
#    xgb_df = xgb_df[[
#        'basement_0', 'basement_1', 'basement_2', 'basement_4',
#        'elevated_building_True', 'contents_location_Lowest_floor_and_above',
#        'contents_location_Upper_floors', 'contents_location_Mobile_home',
#        'nb_floors_2', 'nb_floors_3', 'non_profit_True', 'occupancy_Mobile_home',
#        'occupancy_Multi_unit_residential', 'occupancy_Single_family_residential',
#        'primary_residence_True', 'damage_cause_1', 'damage_cause_2',
#        'damage_cause_3', 'damage_cause_4', 'damage_cause_7', 'damage_cause_9',
#        'damage_cause_A', 'rental_True', 'building_age',
#        'flood_zone_Low_moderate_zone', 'flood_zone_Other_A_zone',
#        'flood_zone_V_zone'
#    ]]
#    new_names = ['basement0', 'basement1', 'basement2', 'basement4',
#       'elevated_buildingTRUE', 'contents_locationLowest_floor_and_above',
#       'contents_locationUpper_floors', 'contents_locationMobile_home',
#       'nb_floors2', 'nb_floors3', 'non_profitTRUE', 'occupancyMobile_home',
#       'occupancyMulti_unit_residential', 'occupancySingle_family_residential',
#       'primary_residenceTRUE', 'damage_cause1', 'damage_cause2',
#       'damage_cause3', 'damage_cause4', 'damage_cause7', 'damage_cause9',
#       'damage_causeA', 'rentalTRUE', 'building_age',
#       'flood_zoneLow_moderate_zone', 'flood_zoneOther_A_zone',
#       'flood_zoneV_zone']
#    xgb_df.columns = new_names
    
#    return xgb_df

In [14]:
#X_test = make_xgb_df(df_test.drop(columns=["Y"]))

In [15]:
class XGBWrapper:
    def predict(self, X):
        return xgb_model.predict(X)

wrapped_model = XGBWrapper()

## DiCE on the 1st query instance

Documentation of the Python package: https://interpret.ml/DiCE/notebooks/DiCE_getting_started.html

In [16]:
continuous_features = X_train.columns.tolist()
X_train["Y"] = df_train["Y"]
# Define the DiCE data object
d = dice_ml.Data(dataframe = X_train, continuous_features = continuous_features, outcome_name = 'Y')

In [17]:
# Define the DiCE model object
m = dice_ml.Model(model = wrapped_model, backend = "sklearn", model_type = 'regressor')

In [18]:
actionable_features = [
    'basement0', 'basement1', 'basement2', 'basement4',
    'elevated_buildingTRUE', 'contents_locationLowest_floor_and_above',
    'contents_locationUpper_floors', 'contents_locationMobile_home'
    ]

In [19]:
# Create explainer
genetic_explainer = dice_ml.Dice(d, m, method = "genetic")
# Genetic method naturally handles categorical variables

In [20]:
cf = genetic_explainer.generate_counterfactuals(
    query_instance,
    total_CFs = 100, # max.
    desired_range=[0, 0.41],  # desired range
    features_to_vary = actionable_features,
    initialization="kdtree",
    proximity_weight=0.2,
    diversity_weight=5.0,
    yloss_type="hinge_loss", 
    diversity_loss_type="dpp_style:inverse_dist",
    maxiterations=10, 
    thresh=1e-2
    )

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:01<00:00,  1.09s/it]


In [21]:
# Show counterfactuals
cf.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0.4582124650478363)


,basement0,basement1,basement2,basement4,elevated_buildingTRUE,contents_locationLowest_floor_and_above,contents_locationUpper_floors,contents_locationMobile_home,nb_floors2,nb_floors3,...,damage_cause4,damage_cause7,damage_cause9,damage_causeA,rentalTRUE,building_age,flood_zoneLow_moderate_zone,flood_zoneOther_A_zone,flood_zoneV_zone,Y
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,64,0,0,0,0.458212



Diverse Counterfactual set (new outcome: [0, 0.41])


,basement0,basement1,basement2,basement4,elevated_buildingTRUE,contents_locationLowest_floor_and_above,contents_locationUpper_floors,contents_locationMobile_home,nb_floors2,nb_floors3,...,damage_cause4,damage_cause7,damage_cause9,damage_causeA,rentalTRUE,building_age,flood_zoneLow_moderate_zone,flood_zoneOther_A_zone,flood_zoneV_zone,Y
0,0.0,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.39981117844581604
0,-,-,-,-,-,-,1.0,-,-,-,...,-,-,-,-,-,-,-,-,-,0.34475457668304443
0,-,1.0,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.3492089509963989
0,-,-,-,-,-,1.0,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.3133501410484314
0,-,-,-,-,1.0,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,0.38275447487831116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,0.0,-,1.0,1.0,-,-,1.0,-,-,-,...,-,-,-,-,-,-,-,-,-,0.3206886947154999
0,-,-,-,1.0,-,1.0,1.0,1.0,-,-,...,-,-,-,-,-,-,-,-,-,0.37932541966438293
0,0.0,-,1.0,-,1.0,-,1.0,-,-,-,...,-,-,-,-,-,-,-,-,-,0.2464744746685028
0,0.0,-,1.0,-,-,1.0,-,1.0,-,-,...,-,-,-,-,-,-,-,-,-,0.3143860995769501


In [22]:
df_cf = cf.cf_examples_list[0].final_cfs_df

In [23]:
df_cf.shape[0]

97

In [24]:
df_cf = df_cf.drop_duplicates().reset_index(drop=True)
df_cf.shape[0]

97

In [25]:
# Basement
cols = [
    "basement0", "basement1", "basement2", "basement4"
]
# We need to ensure that the sum of the one-hot encoded telematics variables equals 1
df_cf = df_cf[df_cf[cols].sum(axis=1).isin([1])]

In [26]:
# Location of contents
cols = [
    "contents_locationLowest_floor_and_above", "contents_locationUpper_floors", "contents_locationMobile_home"
]
# We need to ensure that the sum of the one-hot encoded telematics variables equals 1
df_cf = df_cf[df_cf[cols].sum(axis=1).isin([0, 1])]

In [27]:
df_cf = df_cf.rename(columns={"Y": "pred"})

In [28]:
df_cf.shape[0]

22

In [29]:
df_cf['idx_query'] = 19+1

In [30]:
df_cf.to_csv("../out/flood_cf_dice_first_query.csv", index=False)

## DiCE on the 2nd query instance

In [31]:
# 1st Query instance
idx_query = 3095
query_instance = X_test.iloc[[idx_query]]
df_test.iloc[[idx_query]]

,basement,elevated_building,contents_location,nb_floors,non_profit,occupancy,primary_residence,damage_cause,rental,building_age,flood_zone,Y
3095,4,False,Lowest_floor_only,1,False,Single_family_residential,True,1,False,50,AE_zone,0.798338


In [32]:
cf = genetic_explainer.generate_counterfactuals(
    query_instance,
    total_CFs = 100, # max.
    desired_range=[0, 0.36],  # desired range
    features_to_vary = actionable_features,
    initialization="kdtree",
    proximity_weight=0.2,
    diversity_weight=5.0,
    yloss_type="hinge_loss", 
    diversity_loss_type="dpp_style:inverse_dist",
    maxiterations=10, 
    thresh=1e-2
    )

100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


In [33]:
df_cf = cf.cf_examples_list[0].final_cfs_df
df_cf.shape[0]

94

In [34]:
df_cf = df_cf.drop_duplicates().reset_index(drop=True)
df_cf.shape[0]

94

In [35]:
# Basement
cols = [
    "basement0", "basement1", "basement2", "basement4"
]
# We need to ensure that the sum of the one-hot encoded telematics variables equals 1
df_cf = df_cf[df_cf[cols].sum(axis=1).isin([1])]

In [36]:
# Location of contents
cols = [
    "contents_locationLowest_floor_and_above", "contents_locationUpper_floors", "contents_locationMobile_home"
]
# We need to ensure that the sum of the one-hot encoded telematics variables equals 1
df_cf = df_cf[df_cf[cols].sum(axis=1).isin([0, 1])]

In [37]:
df_cf = df_cf.rename(columns={"Y": "pred"})

In [38]:
df_cf.shape[0]

22

In [39]:
df_cf['idx_query'] = 3095+1

In [40]:
df_cf.to_csv("../out/flood_cf_dice_second_query.csv", index=False)